# **EDA for preprocessing**

**Miêu tả:** Bước này thực hiện EDA chi tiết trước khi tiền xử lý dữ liệu. Code tập trung phân tích phân phối dữ liệu, tỷ lệ và cơ chế missing values như MAR, MCAR, MNAR, phát hiện outlier, kiểm tra dữ liệu trùng lặp, khảo sát khả năng parse các cột văn bản như explain và customer_joined_time, phân tích các cột thời gian như review_created_date, delivery_date, đồng thời phát hiện một số lỗi dữ liệu như format không đồng nhất, giá trị âm và chuỗi văn bản bất thường. Các biểu đồ được tự động lưu vào thư mục eda_preprocessing để làm cơ sở cho bước tiền xử lý và feature engineering tiếp theo.

# **Cài đặt Lib - Config**

In [ ]:
import re
import unicodedata
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

df = pd.read_csv("dataset.csv", low_memory=False)
drop_cols = [
    "images",
    "vote_agree",
    "attributes",
    "product_attributes",
    "suggestions",
    "thank_count",
    "delivery_rating",
    "customer_total_thank",
    "new_score",
    "score",
]
df = df.drop(columns=drop_cols, errors="ignore")
df.to_csv("data2.1.csv", index=False)

INPUT_PATH = "data2.1.csv"
OUT_DIR = Path("eda_preprocessing")
OUT_DIR.mkdir(exist_ok=True)

NUMERIC_COLS = [
    "rating", "num_images", "num_vote_agree",
    "customer_total_review", "customer_total_thank",
    "content_length",
]

OUTLIER_COLS = [
    "customer_total_review",
    "customer_total_thank",
    "num_vote_agree",
    "num_images",
]

DATE_COLS = ["review_created_date", "delivery_date"]

TEXT_PARSE_COLS = ["explain", "customer_joined_time"]

SENTINELS = {"nan", "none", "null", "na", ""}

PALETTE = {
    "primary": "#4C72B0",
    "secondary": "#DD8452",
    "accent": "#55A868",
    "danger": "#C44E52",
    "neutral": "#8172B2",
    "muted": "#937860",
}

FINDINGS = {}

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 140,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

def save(fig, name, subdir=""):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(exist_ok=True)
    fig.savefig(d / f"{name}.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    print(f"[saved] {d / name}.png")


def fmt_int(x, _=None):
    return f"{int(x):,}"

# **Đọc Data**

In [ ]:
def load_data(input_path):
    print("Loading data...")

    df_raw = pd.read_csv(input_path, low_memory=False, dtype=str)
    df = df_raw.copy()

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

    numeric_present = [c for c in NUMERIC_COLS if c in df.columns]

    print(f"Shape: {df.shape}")
    print(f"Numeric columns: {numeric_present}")

    return df_raw, df, numeric_present


df_raw, df, numeric_present = load_data(INPUT_PATH)

Loading data...
Shape: (19, 18)
Numeric columns: ['rating', 'num_images', 'num_vote_agree', 'customer_total_review', 'content_length']


# **Phân phối dữ liệu**

In [ ]:
def s1_data_distribution(df, df_raw):
    print("\n[S1] Phân phối dữ liệu")

    fill_rate = (1 - df.isnull().mean()) * 100
    FINDINGS["fill_rate"] = fill_rate.to_dict()

    fig, ax = plt.subplots(figsize=(10, max(4, len(fill_rate) * 0.32)))

    colors = [
        PALETTE["danger"] if v < 50 else PALETTE["secondary"] if v < 90 else PALETTE["accent"]
        for v in fill_rate
    ]

    bars = ax.barh(fill_rate.index, fill_rate.values, color=colors, height=0.6)

    ax.axvline(90, ls="--", lw=1, color=PALETTE["primary"], label="90% fill")
    ax.axvline(50, ls=":", lw=1, color=PALETTE["danger"], label="50% fill")

    ax.set_xlabel("Fill rate (%)")
    ax.set_title("Tỷ lệ dữ liệu có giá trị ở từng cột")
    ax.set_xlim(0, 108)
    ax.legend(fontsize=9)

    for bar, val in zip(bars, fill_rate.values):
        ax.text(
            min(val + 1, 105),
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%",
            va="center",
            fontsize=8,
        )

    fig.tight_layout()
    save(fig, "s1_column_fill_rates", "s1_data_distribution")

    schema_df = pd.DataFrame({
        "Column": df.columns,
        "Dtype(raw)": df_raw.dtypes.astype(str).values,
        "Non-null": df.notna().sum().values,
        "Unique": [df[c].nunique() for c in df.columns],
    })

    fig, ax = plt.subplots(figsize=(10, max(4, len(schema_df) * 0.35)))
    ax.axis("off")

    tbl = ax.table(
        cellText=schema_df.values,
        colLabels=schema_df.columns,
        cellLoc="left",
        loc="center",
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.auto_set_column_width(list(range(len(schema_df.columns))))

    ax.set_title("Schema tổng quan của dữ liệu", fontsize=12, pad=12)

    fig.tight_layout()
    save(fig, "s1_column_schema", "s1_data_distribution")

    FINDINGS["n_rows"] = len(df)
    FINDINGS["n_cols"] = len(df.columns)

s1_data_distribution(df, df_raw)


[S1] Phân phối dữ liệu
[saved] eda_preprocessing\s1_data_distribution\s1_column_fill_rates.png
[saved] eda_preprocessing\s1_data_distribution\s1_column_schema.png


# **Phân tích Missing Values MAR, MCAR, MNAR**

In [ ]:
def s2_missing_values(df, numeric_present):
    print("\n[S2] Phân tích Missing Values")

    miss_pct = df.isnull().mean() * 100
    miss_pct_nonzero = miss_pct[miss_pct > 0].sort_values(ascending=False)

    FINDINGS["miss_pct"] = miss_pct.to_dict()

    mnar_structural = {"delivery_date", "explain", "customer_joined_time"}
    mnar_threshold = 0.50
    mar_corr_thresh = 0.20

    fill_records = []

    for col in df.columns:
        if "_id" in col.lower():
            continue

        n_miss = df[col].isnull().sum()

        if n_miss == 0:
            continue

        miss_rate = n_miss / len(df)
        is_numeric = pd.api.types.is_numeric_dtype(df[col])

        if col in mnar_structural or miss_rate >= mnar_threshold:
            mechanism = "MNAR"
            fill_val = -999 if is_numeric else "Missing"

        else:
            miss_flag = df[col].isnull().astype(float)
            other_num = df.select_dtypes(include="number").drop(
                columns=[col],
                errors="ignore",
            )

            max_corr = (
                other_num.corrwith(miss_flag).abs().max()
                if not other_num.empty
                else 0
            )

            if pd.notna(max_corr) and max_corr >= mar_corr_thresh:
                mechanism = f"MAR (max |corr|={max_corr:.2f})"
                fill_val = (
                    round(int(df[col].median()), 4)
                    if is_numeric
                    else (
                        df[col].mode(dropna=True).iloc[0]
                        if not df[col].mode(dropna=True).empty
                        else "Unknown"
                    )
                )
            else:
                mechanism = "MCAR"
                fill_val = (
                    round(int(df[col].mean()), 4)
                    if is_numeric
                    else (
                        df[col].mode(dropna=True).iloc[0]
                        if not df[col].mode(dropna=True).empty
                        else "Unknown"
                    )
                )

        fill_records.append({
            "column": col,
            "missing_rate": f"{miss_rate * 100:.1f}%",
            "mechanism": mechanism,
            "suggested_fill": str(fill_val),
        })

    fill_df = pd.DataFrame(fill_records)
    FINDINGS["missing_mechanism"] = fill_records

    if not fill_df.empty:
        fig, ax = plt.subplots(figsize=(14, max(3, len(fill_df) * 0.42)))
        ax.axis("off")

        tbl = ax.table(
            cellText=fill_df.values,
            colLabels=fill_df.columns,
            cellLoc="left",
            loc="center",
        )

        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8.5)
        tbl.auto_set_column_width(list(range(len(fill_df.columns))))

        ax.set_title("Phân loại Missing Values: MAR, MCAR, MNAR", fontsize=12, pad=12)

        fig.tight_layout()
        save(fig, "s2_missing_mechanism_table", "s2_missing")

    # Missingness correlation signal cho từng cột có missing, so với các cột numeric khác
    for target_col in ["customer_total_review"]:
        if target_col not in df.columns or df[target_col].isnull().sum() == 0:
            continue

        miss_flag = df[target_col].isnull().astype(float)
        other_num = df.select_dtypes(include="number").drop(
            columns=[target_col],
            errors="ignore",
        )

        if other_num.empty:
            continue

        corr_s = other_num.corrwith(miss_flag).abs().dropna().sort_values(ascending=True)

        if corr_s.empty:
            continue

        fig, ax = plt.subplots(figsize=(9, max(2.5, len(corr_s) * 0.5)))

        ax.barh(corr_s.index, corr_s.values, color=PALETTE["primary"], height=0.5)

        ax.axvline(
            0.10,
            ls="--",
            color=PALETTE["danger"],
            label="|corr| = 0.10",
        )

        ax.set_xlabel("|Pearson corr| với missingness flag")
        ax.set_title(f"Missingness correlation signal for '{target_col}'")
        ax.legend(fontsize=9, loc="lower right")

        fig.tight_layout()
        save(fig, f"s2_missingness_corr_{target_col}", "s2_missing")

        FINDINGS.setdefault("missingness_corr", {})[target_col] = corr_s.to_dict()

    return fill_df


missing_summary = s2_missing_values(df, numeric_present)
missing_summary


[S2] Phân tích Missing Values
[saved] eda_preprocessing\s2_missing\s2_missing_mechanism_table.png


,column,missing_rate,mechanism,suggested_fill
0,content,47.4%,MAR (max |corr|=0.73),"Nhân viên nhiệt tình, cửa hàng chính hãng, giá..."
1,customer_full_name,5.3%,MAR (max |corr|=0.37),Bình Vương
2,seller_name,5.3%,MAR (max |corr|=0.37),HEAD Tân Long Vân
3,review_created_date,5.3%,MAR (max |corr|=0.37),2022-04-24 10:11:00
4,delivery_date,5.3%,MNAR,Missing
5,explain,5.3%,MNAR,Missing


# **Phát hiện Outliers**

In [ ]:
def s4_outlier_analysis(df):
    print("\n[S4] Phát hiện Outlier")

    outlier_stats = {}

    for col in OUTLIER_COLS:
        if col not in df.columns:
            continue

        series = df[col].dropna()

        if series.empty:
            continue

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        p99 = series.quantile(0.99)

        outlier_stats[col] = {
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "p99": p99,
            "median": series.median(),
            "mean": series.mean(),
            "max": series.max(),
        }

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        axes[0].hist(
            series,
            bins=80,
            color=PALETTE["primary"],
            edgecolor="none",
            alpha=0.8,
        )
        axes[0].set_yscale("log")
        axes[0].set_title(f"{col}\nPhân phối đầy đủ")
        axes[0].set_xlabel("Giá trị")
        axes[0].set_ylabel("Số lượng")
        axes[0].axvline(
            p99,
            ls="--",
            color=PALETTE["danger"],
            label=f"P99 = {p99:.1f}",
        )
        axes[0].legend(fontsize=8)

        axes[1].boxplot(
            series,
            vert=True,
            patch_artist=True,
            boxprops=dict(facecolor=PALETTE["secondary"], alpha=0.6),
            medianprops=dict(color="black", lw=2),
            flierprops=dict(marker=".", ms=2, alpha=0.3),
        )
        axes[1].set_title(f"{col}\nBoxplot — IQR = {iqr:.2f}")
        axes[1].set_ylabel("Giá trị")

        quantiles = np.linspace(0.5, 0.999, 200)
        q_vals = series.quantile(quantiles).values

        axes[2].plot(quantiles * 100, q_vals, color=PALETTE["primary"], lw=2)
        axes[2].axhline(
            p99,
            ls="--",
            color=PALETTE["danger"],
            label=f"P99 = {p99:.1f}",
        )
        axes[2].axhline(
            q3 + 1.5 * iqr,
            ls=":",
            color=PALETTE["neutral"],
            label=f"IQR upper = {q3 + 1.5 * iqr:.1f}",
        )

        axes[2].set_xlabel("Percentile")
        axes[2].set_ylabel("Giá trị")
        axes[2].set_title(f"{col}\nQuantile curve")
        axes[2].legend(fontsize=8)

        fig.suptitle(
            f"Outlier profile: {col} | "
            f"median={series.median():.1f}, "
            f"mean={series.mean():.1f}, "
            f"max={series.max():.0f}",
            fontsize=11,
        )

        fig.tight_layout()
        save(fig, f"s4_outlier_{col}", "s4_outliers")

        print(
            f"{col}: Q1={q1:.1f}, Q3={q3:.1f}, "
            f"IQR={iqr:.1f}, P99={p99:.1f}, max={series.max():.0f}"
        )

    # Raw vs percentile-capped distribution cho customer_total_review
    cap_col = "customer_total_review"

    if cap_col in df.columns and cap_col in outlier_stats:
        series = df[cap_col].dropna()
        cap_val = outlier_stats[cap_col]["p99"]
        capped = series.clip(upper=cap_val)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

        axes[0].hist(series, bins=60, color=PALETTE["danger"], edgecolor="none", alpha=0.85)
        axes[0].axvline(cap_val, ls="--", color=PALETTE["danger"], lw=1.5, label=f"cap = {cap_val:.1f}")
        axes[0].set_title("Raw values")
        axes[0].set_xlabel("Value")
        axes[0].set_ylabel("Count")
        axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))
        axes[0].legend(fontsize=9, loc="lower right")
        axes[0].text(
            0.97, 0.95,
            f"mean={series.mean():.1f}\nmedian={series.median():.1f}\nmax={int(series.max())}",
            transform=axes[0].transAxes, ha="right", va="top", fontsize=9,
        )

        axes[1].hist(capped, bins=60, color=PALETTE["accent"], edgecolor="none", alpha=0.85)
        axes[1].axvline(cap_val, ls="--", color=PALETTE["danger"], lw=1.5, label=f"cap = {cap_val:.1f}")
        axes[1].set_title("Percentile-capped values")
        axes[1].set_xlabel("Value")
        axes[1].set_ylabel("Count")
        axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))
        axes[1].legend(fontsize=9, loc="lower right")
        axes[1].text(
            0.97, 0.95,
            f"mean={capped.mean():.1f}\nmedian={capped.median():.1f}\nmax={int(capped.max())}",
            transform=axes[1].transAxes, ha="right", va="top", fontsize=9,
        )

        fig.suptitle(f"'{cap_col}' — raw distribution vs percentile-capped distribution", fontsize=13)
        fig.tight_layout()
        save(fig, f"s4_{cap_col}_raw_vs_capped", "s4_outliers")

    # Outlier magnitude và concentration: % rows vượt P99 upper bound + % giá trị nằm trong top 1%
    magnitude_cols = [c for c in OUTLIER_COLS if c in df.columns]

    if magnitude_cols:
        pct_above_upper = {}
        pct_value_top1 = {}

        for col in magnitude_cols:
            series = df[col].dropna()

            if series.empty:
                continue

            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1
            upper_bound = q3 + 1.5 * iqr

            if iqr == 0:
                pct_above_upper[col] = None
            else:
                pct_above_upper[col] = (series > upper_bound).mean() * 100

            top1_thresh = series.quantile(0.99)
            top1_vals = series[series >= top1_thresh].sum()
            total_vals = series.sum()

            pct_value_top1[col] = (top1_vals / total_vals * 100) if total_vals > 0 else 0.0

        fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

        labels0 = list(pct_above_upper.keys())
        values0 = [v if v is not None else 0 for v in pct_above_upper.values()]

        bars0 = axes[0].bar(labels0, values0, color=PALETTE["accent"])

        axes[0].axhline(8, ls="--", color=PALETTE["danger"], label="8% reference line")
        axes[0].set_ylabel("% rows above upper bound")
        axes[0].set_title(
            "Proportion of rows exceeding the P99 upper bound per column\n"
            "(N/A shown as 0 for columns with IQR = 0)"
        )
        axes[0].legend(fontsize=9)

        for bar, col in zip(bars0, labels0):
            label_text = "N/A" if pct_above_upper[col] is None else f"{pct_above_upper[col]:.2f}%"
            axes[0].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values0, default=1) * 0.03,
                label_text,
                ha="center", fontsize=9,
            )

        labels1 = list(pct_value_top1.keys())
        values1 = list(pct_value_top1.values())

        bars1 = axes[1].bar(labels1, values1, color=PALETTE["neutral"])

        axes[1].set_ylabel("% of total value held by top 1% rows")
        axes[1].set_title("Value concentration in the top 1% of rows per column")

        for bar, v in zip(bars1, values1):
            axes[1].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values1, default=1) * 0.02,
                f"{v:.1f}%",
                ha="center", fontsize=9,
            )

        fig.suptitle("Outlier magnitude and concentration analysis", fontsize=13)
        fig.tight_layout()
        save(fig, "s4_outlier_magnitude_concentration", "s4_outliers")

        FINDINGS["outlier_magnitude"] = {
            "pct_above_upper_bound": pct_above_upper,
            "pct_value_top1": pct_value_top1,
        }

    FINDINGS["outlier_stats"] = outlier_stats

    return pd.DataFrame(outlier_stats).T


outlier_summary = s4_outlier_analysis(df)
outlier_summary


[S4] Phát hiện Outlier
[saved] eda_preprocessing\s4_outliers\s4_outlier_customer_total_review.png
customer_total_review: Q1=2.5, Q3=21.0, IQR=18.5, P99=44.6, max=45
[saved] eda_preprocessing\s4_outliers\s4_outlier_num_vote_agree.png
num_vote_agree: Q1=0.0, Q3=0.0, IQR=0.0, P99=4.0, max=4
[saved] eda_preprocessing\s4_outliers\s4_outlier_num_images.png
num_images: Q1=0.0, Q3=0.5, IQR=0.5, P99=1.0, max=1


,q1,q3,iqr,p99,median,mean,max
customer_total_review,2.5,21.0,18.5,44.64,12.0,14.947368,45.0
num_vote_agree,0.0,0.0,0.0,4.00,0.0,0.631579,4.0
num_images,0.0,0.5,0.5,1.00,0.0,0.263158,1.0


# **Phân tích Duplication**

In [ ]:
def s5_duplication_analysis(df):
    print("\n[S5] Phân tích Duplication")

    duplicate_results = {}

    if "review_id" in df.columns:
        dup_rid = df.duplicated(subset=["review_id"], keep=False).sum()
        unique_rid = df["review_id"].nunique()

        duplicate_results["duplicate_review_id"] = int(dup_rid)

        fig, ax = plt.subplots(figsize=(6, 3.5))

        ax.bar(
            ["Unique review_id", "Rows sharing review_id"],
            [unique_rid, dup_rid],
            color=[PALETTE["accent"], PALETTE["danger"]],
        )

        ax.set_title("Kiểm tra trùng lặp review_id")
        ax.set_ylabel("Số dòng")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

        for bar in ax.patches:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + len(df) * 0.003,
                f"{int(bar.get_height()):,}",
                ha="center",
                fontsize=10,
            )

        fig.tight_layout()
        save(fig, "s5_duplicate_review_id", "s5_duplicates")

    key_cols = ["customer_id", "product_id", "content"]

    if all(c in df.columns for c in key_cols):
        has_content = df["content"].notna()

        dup_cpc = df[has_content].duplicated(
            subset=key_cols,
            keep=False,
        ).sum()

        duplicate_results["duplicate_customer_product_content"] = int(dup_cpc)

        fig, ax = plt.subplots(figsize=(6, 3.5))

        ax.bar(
            ["Non-duplicate", "Same customer-product-content"],
            [has_content.sum() - dup_cpc, dup_cpc],
            color=[PALETTE["accent"], PALETTE["danger"]],
        )

        ax.set_title("Trùng lặp theo customer_id, product_id và content")
        ax.set_ylabel("Số dòng")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

        for bar in ax.patches:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + has_content.sum() * 0.003,
                f"{int(bar.get_height()):,}",
                ha="center",
                fontsize=9,
            )

        fig.tight_layout()
        save(fig, "s5_duplicate_customer_product_content", "s5_duplicates")

        def normalise(s):
            return (
                s.astype(str)
                .apply(lambda t: unicodedata.normalize("NFC", t))
                .str.lower()
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )

        subset = df[has_content].copy()
        subset["_norm"] = normalise(subset["content"])

        dup_norm = subset.duplicated(
            subset=["customer_id", "product_id", "_norm"],
            keep=False,
        ).sum()

    # Phân tích review count per customer / seller / product: histogram (log y),
    # quantile curve và tỷ lệ review do top 1% id chiếm giữ
    id_cols = [
        ("customer_id", "customer"),
        ("seller_id", "seller"),
        ("product_id", "product"),
    ]

    concentration_results = {}

    for id_col, label in id_cols:
        if id_col not in df.columns:
            continue

        counts = df[id_col].dropna().value_counts()

        if counts.empty:
            continue

        n_unique = len(counts)
        median_c = counts.median()
        mean_c = counts.mean()
        max_c = counts.max()

        top1_n = max(1, int(np.ceil(n_unique * 0.01)))
        sorted_counts = counts.sort_values(ascending=False)
        top1_reviews = int(sorted_counts.iloc[:top1_n].sum())
        rest_reviews = int(sorted_counts.iloc[top1_n:].sum())
        total_reviews = top1_reviews + rest_reviews
        top1_share = top1_reviews / total_reviews * 100 if total_reviews else 0.0

        concentration_results[id_col] = {
            "unique": int(n_unique),
            "median": float(median_c),
            "mean": float(mean_c),
            "max": int(max_c),
            "top1_reviews": top1_reviews,
            "rest_reviews": rest_reviews,
            "top1_share_pct": top1_share,
        }

        fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

        axes[0].hist(counts.values, bins=60, color=PALETTE["primary"], edgecolor="none", alpha=0.85)
        axes[0].set_yscale("log")
        axes[0].set_title(f"Reviews per {label}\nHistogram (log y)")
        axes[0].set_xlabel(f"Review count per {id_col}")
        axes[0].set_ylabel("Frequency (log y)")

        quantiles = np.linspace(0.5, 1.0, 200)
        q_vals = counts.quantile(quantiles).values
        p90, p95, p99 = counts.quantile([0.90, 0.95, 0.99])

        axes[1].plot(quantiles * 100, q_vals, color=PALETTE["primary"], lw=2)
        axes[1].axhline(p90, ls=":", color=PALETTE["muted"], label=f"P90 = {p90:.0f}")
        axes[1].axhline(p95, ls="--", color=PALETTE["muted"], label=f"P95 = {p95:.0f}")
        axes[1].axhline(p99, ls="-.", color=PALETTE["muted"], label=f"P99 = {p99:.0f}")
        axes[1].set_xlabel("Percentile")
        axes[1].set_ylabel("Review count")
        axes[1].set_title(f"Reviews per {label}\nQuantile curve")
        axes[1].legend(fontsize=8)

        axes[2].pie(
            [top1_reviews, rest_reviews],
            labels=[
                f"Top 1% of {id_col}s\n({top1_reviews:,} reviews)",
                f"Bottom 99%\n({rest_reviews:,} reviews)",
            ],
            colors=[PALETTE["danger"], PALETTE["accent"]],
            autopct="%1.1f%%",
            startangle=90,
        )
        axes[2].set_title(f"Reviews per {label}\nReview share: top 1% vs rest")

        fig.suptitle(
            f"Reviews per {label} | unique: {n_unique:,} | "
            f"median={median_c:.0f}, mean={mean_c:.1f}, max={max_c:,}",
            fontsize=12,
        )

        fig.tight_layout()
        save(fig, f"s5_reviews_per_{label}", "s5_concentration")

        print(
            f"{id_col}: unique={n_unique:,}, median={median_c:.0f}, "
            f"mean={mean_c:.1f}, max={max_c:,}, top1% share={top1_share:.1f}%"
        )

    FINDINGS["review_concentration"] = concentration_results

    # No-content rate per seller
    if "seller_id" in df.columns and "content" in df.columns:
        has_content = df["content"].notna()
        no_content_rate_per_seller = (
            (~has_content).groupby(df["seller_id"]).mean() * 100
        )

        fig, ax = plt.subplots(figsize=(12, 5))

        ax.hist(
            no_content_rate_per_seller.values,
            bins=50,
            color=PALETTE["secondary"],
            edgecolor="none",
            alpha=0.9,
        )

        ax.set_xlabel("No-content rate per seller (%)")
        ax.set_ylabel("Number of sellers")
        ax.set_title(
            "Distribution of no-content review rate per seller\n"
            "(one bar = one seller; x-axis = % of their reviews with no text)"
        )

        fig.tight_layout()
        save(fig, "s5_no_content_rate_per_seller", "s5_concentration")

        FINDINGS["no_content_rate_per_seller"] = no_content_rate_per_seller.describe().to_dict()

    # No-content rate per star rating
    if "rating" in df.columns and "content" in df.columns:
        has_content = df["content"].notna()
        no_content_rate_per_rating = (
            (~has_content).groupby(df["rating"]).mean() * 100
        ).sort_index()

        fig, ax = plt.subplots(figsize=(10, 4.5))

        bars = ax.bar(
            no_content_rate_per_rating.index.astype(str),
            no_content_rate_per_rating.values,
            color=PALETTE["danger"],
        )

        ax.set_xlabel("Star rating")
        ax.set_ylabel("No-content rate (%)")
        ax.set_title("No-content rate per star rating")

        for bar, v in zip(bars, no_content_rate_per_rating.values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(no_content_rate_per_rating.values, default=1) * 0.01,
                f"{v:.1f}%",
                ha="center", fontsize=10,
            )

        fig.tight_layout()
        save(fig, "s5_no_content_rate_per_rating", "s5_concentration")

        FINDINGS["no_content_rate_per_rating"] = no_content_rate_per_rating.to_dict()

    # Content length distribution (chỉ các dòng có content, clip tại P99)
    if "content" in df.columns:
        content_len = df.loc[df["content"].notna(), "content"].astype(str).str.len()

        if not content_len.empty:
            p99_len = content_len.quantile(0.99)
            clipped_len = content_len.clip(upper=p99_len)

            fig, ax = plt.subplots(figsize=(11, 4.5))

            ax.hist(
                clipped_len,
                bins=50,
                color=PALETTE["primary"],
                edgecolor="none",
                alpha=0.9,
            )

            ax.set_xlabel("Content length (characters)")
            ax.set_ylabel("Count")
            ax.set_title("Content length distribution — rows with non-null content (clipped at P99)")
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

            fig.tight_layout()
            save(fig, "s5_content_length_distribution", "s5_concentration")

            FINDINGS["content_length_stats"] = content_len.describe().to_dict()

    FINDINGS["duplication"] = duplicate_results

    return duplicate_results


duplicate_summary = s5_duplication_analysis(df)
duplicate_summary


[S5] Phân tích Duplication
[saved] eda_preprocessing\s5_duplicates\s5_duplicate_review_id.png
[saved] eda_preprocessing\s5_duplicates\s5_duplicate_customer_product_content.png


{'duplicate_review_id': 0, 'duplicate_customer_product_content': 0}

# **Khảo sát khả năng parse cột văn bản**

In [ ]:
UNIT_PATTERN = re.compile(
    r"(\d+(?:\.\d+)?)\s*(nam|thang|ngay|gio|phut|năm|tháng|ngày|giờ|phút)",
    re.IGNORECASE,
)

EXPLAIN_HOURS = {
    "nam": 8640.0,
    "năm": 8640.0,
    "thang": 720.0,
    "tháng": 720.0,
    "ngay": 24.0,
    "ngày": 24.0,
    "gio": 1.0,
    "giờ": 1.0,
    "phut": 1 / 60,
    "phút": 1 / 60,
}

JOINED_MONTHS = {
    "nam": 12.0,
    "năm": 12.0,
    "thang": 1.0,
    "tháng": 1.0,
    "ngay": 0.0,
    "ngày": 0.0,
}


def s6_text_parsing(df):
    print("\n[S6] Parse các cột text")

    parse_stats = {}

    for col in TEXT_PARSE_COLS:
        if col not in df.columns:
            continue

        raw = df[col].dropna().astype(str)

        if raw.empty:
            continue

        unit_counts = {}
        parseable = 0

        for text in raw:
            m = UNIT_PATTERN.search(text)

            if m:
                parseable += 1
                unit = m.group(2).lower()
                unit_counts[unit] = unit_counts.get(unit, 0) + 1

        parse_rate = parseable / len(raw) * 100

        parse_stats[col] = {
            "parseable": parseable,
            "total": len(raw),
            "parse_rate_pct": parse_rate,
            "units": unit_counts,
        }

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        not_parseable = len(raw) - parseable

        bars0 = axes[0].bar(
            ["Parseable\n(regex matched)", "Not parseable\n(no match found)"],
            [parseable, not_parseable],
            color=[PALETTE["accent"], PALETTE["danger"]],
        )

        axes[0].set_title(f"'{col}' — parseability (time-unit regex)")
        axes[0].set_ylabel("Count")
        axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

        for bar in bars0:
            axes[0].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(parseable, not_parseable, 1) * 0.01,
                f"{int(bar.get_height()):,}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

        if unit_counts:
            unit_s = pd.Series(unit_counts).sort_values(ascending=False)

            bars1 = axes[1].bar(
                unit_s.index,
                unit_s.values,
                color=PALETTE["primary"],
            )

            axes[1].set_title(f"'{col}' — time unit frequency")
            axes[1].set_ylabel("Count")
            axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

            for bar in bars1:
                axes[1].text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + unit_s.values.max() * 0.01,
                    f"{int(bar.get_height()):,}",
                    ha="center",
                    va="bottom",
                    fontsize=9,
                )

        fig.suptitle(
            f"'{col}' — {parseable:,} parseable / {len(raw):,} non-null ({parse_rate:.1f}%)",
            fontsize=13,
        )

        fig.tight_layout()
        save(fig, f"s6_parse_{col}", "s6_text_parsing")

        conv = EXPLAIN_HOURS if col == "explain" else JOINED_MONTHS
        label = "Usage hours" if col == "explain" else "Membership months"

        vals = []

        for text in raw:
            m = UNIT_PATTERN.search(text)

            if m:
                unit = m.group(2).lower()

                if unit in conv:
                    vals.append(float(m.group(1)) * conv[unit])

        if vals:
            s = pd.Series(vals)

            fig, ax = plt.subplots(figsize=(9, 4))

            ax.hist(
                s.clip(upper=s.quantile(0.99)),
                bins=50,
                color=PALETTE["neutral"],
                edgecolor="none",
                alpha=0.85,
            )

            ax.set_xlabel(f"{label} sau khi quy đổi")
            ax.set_ylabel("Số lượng")
            ax.set_title(f"{col} — phân phối sau khi parse và quy đổi")

            fig.tight_layout()
            save(fig, f"s6_{col}_numeric_distribution", "s6_text_parsing")

    FINDINGS["parse_stats"] = parse_stats

    return parse_stats


parse_summary = s6_text_parsing(df)
parse_summary


[S6] Parse các cột text
[saved] eda_preprocessing\s6_text_parsing\s6_parse_explain.png
[saved] eda_preprocessing\s6_text_parsing\s6_explain_numeric_distribution.png
[saved] eda_preprocessing\s6_text_parsing\s6_parse_customer_joined_time.png
[saved] eda_preprocessing\s6_text_parsing\s6_customer_joined_time_numeric_distribution.png


{'explain': {'parseable': 18,
  'total': 18,
  'parse_rate_pct': 100.0,
  'units': {'tháng': 5, 'giờ': 3, 'ngày': 7, 'phút': 2, 'năm': 1}},
 'customer_joined_time': {'parseable': 19,
  'total': 19,
  'parse_rate_pct': 100.0,
  'units': {'năm': 19}}}

# **Phân tích temporal feature**

In [ ]:
def s7_temporal_analysis(df):
    print("\n[S7] Phân tích cột thời gian")

    temporal_results = {}

    for date_col in DATE_COLS:
        if date_col not in df.columns:
            continue

        dated = df[date_col].dropna()
        n_dated = len(dated)
        n_undated = len(df) - n_dated

        temporal_results[date_col] = {
            "has_date": n_dated,
            "missing_date": n_undated,
            "missing_pct": n_undated / len(df) * 100,
        }

        if dated.empty:
            continue

        monthly = dated.dt.to_period("M").value_counts().sort_index()

        fig, ax = plt.subplots(figsize=(14, 4))

        ax.bar(
            monthly.index.astype(str),
            monthly.values,
            color=PALETTE["primary"],
            width=0.8,
            alpha=0.85,
        )

        ax.set_xlabel("Tháng")
        ax.set_ylabel("Số lượng review")
        ax.set_title(
            f"{date_col} — phân phối theo tháng | "
            f"có ngày: {n_dated:,}, thiếu ngày: {n_undated:,}"
        )

        step = max(1, len(monthly) // 20)

        ax.set_xticks(range(0, len(monthly), step))
        ax.set_xticklabels(
            [str(monthly.index[i]) for i in range(0, len(monthly), step)],
            rotation=45,
            ha="right",
            fontsize=8,
        )

        fig.tight_layout()
        save(fig, f"s7_monthly_{date_col}", "s7_temporal")

    # Phát hiện customer có >=5 review trong cùng 1 ngày (dấu hiệu bất thường)
    if "customer_id" in df.columns and "review_created_date" in df.columns:
        dated_df = df.dropna(subset=["review_created_date", "customer_id"])

        daily_counts = dated_df.groupby(
            [dated_df["customer_id"], dated_df["review_created_date"].dt.date]
        ).size()

        burst_customers = daily_counts[daily_counts >= 5].index.get_level_values(0).unique()

        n_burst = len(burst_customers)
        n_total_customers = df["customer_id"].nunique()
        burst_pct = n_burst / n_total_customers * 100 if n_total_customers else 0.0

        n_standard = n_total_customers - n_burst

        fig, ax = plt.subplots(figsize=(7, 4.5))

        bars = ax.bar(
            ["Standard customers", "≥5 reviews in one day"],
            [n_standard, n_burst],
            color=[PALETTE["accent"], PALETTE["danger"]],
        )

        ax.set_ylabel("Customer count")
        ax.set_title(
            f"Customers with ≥5 reviews in a single calendar day\n"
            f"{n_burst:,} customers ({burst_pct:.1f}%)"
        )
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + n_standard * 0.005,
                f"{int(bar.get_height()):,}",
                ha="center", fontsize=10,
            )

        fig.tight_layout()
        save(fig, "s7_customers_review_burst_per_day", "s7_temporal")

        temporal_results["customer_review_burst"] = {
            "n_burst_customers": int(n_burst),
            "n_total_customers": int(n_total_customers),
            "burst_pct": burst_pct,
        }

        print(f"Customers with >=5 reviews in one day: {n_burst:,} ({burst_pct:.1f}%)")

    FINDINGS["temporal"] = temporal_results

    return temporal_results


temporal_summary = s7_temporal_analysis(df)
temporal_summary


[S7] Phân tích cột thời gian
[saved] eda_preprocessing\s7_temporal\s7_monthly_review_created_date.png
[saved] eda_preprocessing\s7_temporal\s7_monthly_delivery_date.png


{'review_created_date': {'has_date': 18,
  'missing_date': 1,
  'missing_pct': 5.263157894736842},
 'delivery_date': {'has_date': 18,
  'missing_date': 1,
  'missing_pct': 5.263157894736842}}

# **Phát hiện lỗi**

In [ ]:
def s9_data_error_detection(df, df_raw):
    print("\n[S9] Phát hiện lỗi dữ liệu")

    error_results = {}

    # Phát hiện embedded header / format không đồng nhất ở cột rating
    if "rating" in df_raw.columns:
        header_mask = df_raw["rating"].astype(str).str.strip() == "rating"
        n_headers = int(header_mask.sum())
        n_normal = len(df_raw) - n_headers

        error_results["embedded_header_rows"] = n_headers

        fig, ax = plt.subplots(figsize=(6, 3.5))

        ax.bar(
            ["Normal rows", 'rating == "rating"'],
            [n_normal, n_headers],
            color=[PALETTE["accent"], PALETTE["danger"]],
        )

        ax.set_title("Dòng lỗi do header bị lẫn vào dữ liệu")
        ax.set_ylabel("Số dòng")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

        for bar in ax.patches:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + n_normal * 0.003,
                f"{int(bar.get_height()):,}",
                ha="center",
                fontsize=10,
            )

        fig.tight_layout()
        save(fig, "s9_embedded_header_rows", "s9_data_errors")

        rating_num = pd.to_numeric(df_raw["rating"], errors="coerce")
        invalid_rating = (~rating_num.isin([1, 2, 3, 4, 5]) & rating_num.notna()).sum()
        rating_nan = rating_num.isna().sum()

        error_results["invalid_rating"] = int(invalid_rating)
        error_results["rating_nan"] = int(rating_nan)

        print(f"Embedded header rows: {n_headers}")
        print(f"Invalid rating values: {invalid_rating}")
        print(f"NaN rating values: {rating_nan}")

    # Phát hiện giá trị âm ở các cột dạng count
    non_negative_cols = [
        "num_images",
        "num_vote_agree",
        "customer_total_review",
        "customer_total_thank",
    ]

    neg_counts = {}

    for col in non_negative_cols:
        if col not in df.columns:
            continue

        neg_counts[col] = int((df[col] < 0).sum())
        print(f"{col}: {neg_counts[col]} negative values")

    error_results["negative_values"] = neg_counts

    if neg_counts:
        fig, ax = plt.subplots(figsize=(8, 3.5))

        color = PALETTE["danger"] if any(v > 0 for v in neg_counts.values()) else PALETTE["accent"]

        ax.bar(
            neg_counts.keys(),
            neg_counts.values(),
            color=color,
        )

        ax.set_title("Giá trị âm ở các cột dạng số đếm")
        ax.set_ylabel("Số dòng lỗi")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

        for bar in ax.patches:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(1, max(neg_counts.values(), default=1) * 0.02),
                f"{int(bar.get_height()):,}",
                ha="center",
                fontsize=10,
            )

        fig.tight_layout()
        save(fig, "s9_negative_impossible_values", "s9_data_errors")

    # Phát hiện text bất thường / chuỗi giả null trong content
    if "content" in df_raw.columns:
        raw_content = df_raw["content"].fillna("").astype(str).str.strip().str.lower()

        sentinel_counts = {}

        for s in SENTINELS:
            label = f'"{s}"' if s else '"" empty string'
            sentinel_counts[label] = int((raw_content == s).sum())

        sentinel_counts["NaN original null"] = int(df_raw["content"].isnull().sum())

        sc = pd.Series(sentinel_counts).sort_values(ascending=False)
        sc = sc[sc > 0]

        error_results["sentinel_strings_content"] = sc.to_dict()

        if not sc.empty:
            fig, ax = plt.subplots(figsize=(9, max(3, len(sc) * 0.45)))

            ax.barh(
                sc.index,
                sc.values,
                color=PALETTE["neutral"],
                height=0.6,
            )

            ax.set_xlabel("Số lượng")
            ax.set_title("Chuỗi giả null / text bất thường trong content")
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

            for i, val in enumerate(sc.values):
                ax.text(
                    val + max(1, sc.max() * 0.005),
                    i,
                    f"{val:,}",
                    va="center",
                    fontsize=9,
                )

            fig.tight_layout()
            save(fig, "s9_sentinel_strings_content", "s9_data_errors")

        print(f"Sentinel strings in content: {dict(sc)}")

    # Phát hiện lỗi parse ngày
    date_parse_errors = {}

    for col in DATE_COLS:
        if col not in df_raw.columns:
            continue

        raw_non_null = df_raw[col].notna().sum()
        parsed_non_null = df[col].notna().sum()
        failed_parse = raw_non_null - parsed_non_null

        date_parse_errors[col] = {
            "raw_non_null": int(raw_non_null),
            "parsed_non_null": int(parsed_non_null),
            "failed_parse": int(failed_parse),
        }

        print(f"{col}: failed parse = {failed_parse:,}")

    error_results["date_parse_errors"] = date_parse_errors

    FINDINGS["data_errors"] = error_results

    return error_results


error_summary = s9_data_error_detection(df, df_raw)
error_summary


[S9] Phát hiện lỗi dữ liệu
[saved] eda_preprocessing\s9_data_errors\s9_embedded_header_rows.png
Embedded header rows: 0
Invalid rating values: 0
NaN rating values: 0
num_images: 0 negative values
num_vote_agree: 0 negative values
customer_total_review: 0 negative values
[saved] eda_preprocessing\s9_data_errors\s9_negative_impossible_values.png
[saved] eda_preprocessing\s9_data_errors\s9_sentinel_strings_content.png
Sentinel strings in content: {'"" empty string': np.int64(9), 'NaN original null': np.int64(9)}
review_created_date: failed parse = 0
delivery_date: failed parse = 0


{'embedded_header_rows': 0,
 'invalid_rating': 0,
 'rating_nan': 0,
 'negative_values': {'num_images': 0,
  'num_vote_agree': 0,
  'customer_total_review': 0},
 'sentinel_strings_content': {'"" empty string': 9, 'NaN original null': 9},
 'date_parse_errors': {'review_created_date': {'raw_non_null': 18,
   'parsed_non_null': 18,
   'failed_parse': 0},
  'delivery_date': {'raw_non_null': 18,
   'parsed_non_null': 18,
   'failed_parse': 0}}}

# **Phân phối giá trị Rating**

In [ ]:
def s10_rating_value_distribution(df_raw):
    print("\n[S10] Phân phối giá trị Rating")

    rating_num = pd.to_numeric(df_raw["rating"], errors="coerce")

    valid_vals = [1.0, 2.0, 3.0, 4.0, 5.0]
    labels = [str(v) for v in valid_vals] + ["nan"]

    counts = [int((rating_num == v).sum()) for v in valid_vals]
    counts.append(int(rating_num.isna().sum()))

    fig, ax = plt.subplots(figsize=(9, 5))

    bars = ax.bar(labels, counts, color=PALETTE["danger"], width=0.6)

    ax.set_title("Rating value distribution")
    ax.set_xlabel("Rating value")
    ax.set_ylabel("Count")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_int))

    for bar, c in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            f"{c:,}",
            ha="left",
            va="bottom",
            fontsize=9,
            rotation=45,
        )

    fig.tight_layout()
    save(fig, "s10_rating_value_distribution", "s10_rating_distribution")

    rating_dist = dict(zip(labels, counts))
    FINDINGS["rating_value_distribution"] = rating_dist

    print(f"Rating distribution: {rating_dist}")

    return rating_dist


rating_distribution = s10_rating_value_distribution(df_raw)
rating_distribution
